INSTALL PACKAGES

In [ ]:
!pip install xgboost
!pip install scikit-learn
!pip install joblib
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn

UPLOAD DATASET

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Astram event data_anonymized - Astram event data_anonymizedb40ac87 (1).csv to Astram event data_anonymized - Astram event data_anonymizedb40ac87 (1).csv


LOAD DATA

In [ ]:
import pandas as pd

df = pd.read_csv(
    "Astram event data_anonymized - Astram event data_anonymizedb40ac87 (1) (1).csv"
)

print(df.shape)

df.head()

(8173, 46)


,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


COLUMN CHECK

In [ ]:
print(df.columns.tolist())

['id', 'event_type', 'latitude', 'longitude', 'endlatitude', 'endlongitude', 'address', 'end_address', 'event_cause', 'requires_road_closure', 'start_datetime', 'end_datetime', 'status', 'authenticated', 'modified_datetime', 'map_file', 'direction', 'description', 'veh_type', 'veh_no', 'corridor', 'priority', 'cargo_material', 'reason_breakdown', 'age_of_truck', 'created_date', 'route_path', 'client_id', 'created_by_id', 'last_modified_by_id', 'assigned_to_police_id', 'citizen_accident_id', 'comment', 'police_station', 'meta_data', 'kgid', 'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude', 'closed_by_id', 'closed_datetime', 'resolved_by_id', 'resolved_datetime', 'gba_identifier', 'zone', 'junction']


CLEAN DATA

In [ ]:
import numpy as np

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

datetime_cols = [

    "start_datetime",
    "end_datetime",
    "created_date",
    "modified_datetime",
    "closed_datetime",
    "resolved_datetime"

]

for col in datetime_cols:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

bool_cols = [

    "requires_road_closure",
    "authenticated"

]

for col in bool_cols:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .map({

                "true":1,
                "false":0,

                "yes":1,
                "no":0

            })
        )

numeric_cols = [

    "latitude",
    "longitude",
    "endlatitude",
    "endlongitude",
    "age_of_truck"

]

for col in numeric_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

df = df.drop_duplicates()

print(df.shape)

(8173, 46)


DURATION FEATURE

In [ ]:
df["duration_minutes"] = (

    (
        df["end_datetime"]
        -
        df["start_datetime"]
    )

    .dt.total_seconds()

    / 60

)


In [ ]:
df["duration_minutes"] = (

    df["duration_minutes"]

    .clip(lower=0)

)

print(

    df["duration_minutes"]

    .describe()

)

count    3.820000e+02
mean     9.076910e+03
std      1.077379e+05
min      0.000000e+00
25%      2.267883e+00
50%      3.250827e+02
75%      7.824316e+02
max      2.051059e+06
Name: duration_minutes, dtype: float64


In [ ]:
# cap insane durations

df["duration_minutes"] = (

    df["duration_minutes"]

    .clip(upper=1440)

)

print(

    df["duration_minutes"]

    .describe()

)

count     382.000000
mean      518.602839
std       548.916377
min         0.000000
25%         2.267883
50%       325.082700
75%       782.431629
max      1440.000000
Name: duration_minutes, dtype: float64


In [ ]:
df["duration_available"] = (

    df["duration_minutes"] > 0

).astype(int)

print(

    df["duration_available"]

    .value_counts()

)

duration_available
0    7839
1     334
Name: count, dtype: int64


In [ ]:
event_duration = (

    df[df["duration_minutes"] > 0]

    .groupby("event_type")

    ["duration_minutes"]

    .median()

)

event_duration

,duration_minutes
event_type,
planned,472.30945
unplanned,1440.00000


In [ ]:
df["duration_minutes"] = (

    df.apply(

        lambda row:

        event_duration.get(

            row["event_type"],

            30

        )

        if row["duration_minutes"] == 0

        else row["duration_minutes"],

        axis=1

    )

)

In [ ]:
print(

    df["duration_minutes"]

    .describe()

)

count     382.000000
mean      583.017066
std       517.668884
min         0.028550
25%        85.007671
50%       472.309450
75%       784.218100
max      1440.000000
Name: duration_minutes, dtype: float64


In [ ]:
print(df.shape)

(8173, 48)


TIME FEATURES

In [ ]:
df["hour"] = df["start_datetime"].dt.hour

df["weekday"] = df["start_datetime"].dt.weekday

df["month"] = df["start_datetime"].dt.month

df["weekend"] = (

    df["weekday"]

    .isin([5,6])

).astype(int)

df["peak_hour"] = (

    df["hour"]

    .isin(

        [8,9,10,17,18,19]

    )

).astype(int)

EVENT SEVERITY TABLE

EVENT SEVERITY SCORE

In [ ]:
event_freq = (

    df["event_type"]

    .value_counts(
        normalize=True
    )

)

df["event_frequency_score"] = (

    df["event_type"]

    .map(event_freq)

)

CLOSURE RISK

In [ ]:
closure_risk = (

    df.groupby(
        "event_type"
    )

    ["requires_road_closure"]

    .mean()

)

df["closure_risk"] = (

    df["event_type"]

    .map(
        closure_risk
    )

)

Duration Risk

In [ ]:
duration_risk = (

    df.groupby(
        "event_type"
    )

    ["duration_minutes"]

    .median()

)

df["duration_risk"] = (

    df["event_type"]

    .map(
        duration_risk
    )

)

JUNCTION RISK

In [ ]:
junction_risk = {}

for j in df["junction"].dropna().unique():

    subset = df[df["junction"] == j]

    mean_duration = subset["duration_minutes"].fillna(0).mean()

    score = (
        0.5 * len(subset)
        +
        0.5 * mean_duration
    )

    junction_risk[j] = float(score)

In [ ]:
sum(pd.isna(list(junction_risk.values())))

np.int64(0)

CORRIDOR RISK

In [ ]:
corridor_risk = {}

for c in df["corridor"].dropna().unique():

    subset = df[df["corridor"] == c]

    mean_duration = subset["duration_minutes"].fillna(0).mean()

    score = (
        0.5 * len(subset)
        +
        0.5 * mean_duration
    )

    corridor_risk[c] = float(score)

In [ ]:
sum(pd.isna(list(corridor_risk.values())))

np.int64(0)

HISTORICAL IMPACT

In [ ]:
historical = (

    df.groupby(

        [

            "event_type",

            "corridor",

            "hour"

        ]

    )

    ["duration_minutes"]

    .median()

)

In [ ]:
df["historical_impact"] = (

    df.set_index(

        [

            "event_type",

            "corridor",

            "hour"

        ]

    )

    .index

    .map(
        historical
    )

)

In [ ]:
junction_count = (
    df["junction"]
    .value_counts()
)

junction_duration = (
    df.groupby("junction")
    ["duration_minutes"]
    .mean()
)

corridor_count = (
    df["corridor"]
    .value_counts()
)

corridor_duration = (
    df.groupby("corridor")
    ["duration_minutes"]
    .mean()
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Ensure the 'count' and 'duration' features for junctions and corridors are in the DataFrame
# These Series (junction_count, junction_duration, corridor_count, corridor_duration)
# are expected to be available in the kernel from previous calculations.
df["junction_count"] = df["junction"].map(junction_count)
df["junction_duration"] = df["junction"].map(junction_duration)
df["corridor_count"] = df["corridor"].map(corridor_count)
df["corridor_duration"] = df["corridor"].map(corridor_duration)

risk_cols = [
    "event_frequency_score",
    "closure_risk",
    "duration_risk",
    "junction_count",
    "junction_duration",
    "corridor_count",
    "corridor_duration",
    "historical_impact"
]

# Impute NaN values in the risk columns with their median before scaling.
# This ensures MinMaxScaler can process the data without errors.
for col in risk_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

scaler = MinMaxScaler()

df[risk_cols] = scaler.fit_transform(
    df[risk_cols]
)

BUILD DATA-DRIVEN TII

In [ ]:
from sklearn.preprocessing import MinMaxScaler

tii_scaler = MinMaxScaler(
    feature_range=(0,100)
)

df["traffic_impact_index"] = (

    tii_scaler.fit_transform(

        df[risk_cols]

        .mean(axis=1)

        .values

        .reshape(-1,1)

    )

)

In [ ]:
df["traffic_impact_index"] = (

    df["traffic_impact_index"]

    .astype(float)

)

In [ ]:
print(

    df["traffic_impact_index"]

    .describe()

)

count    8173.000000
mean       68.421655
std        13.704989
min         0.000000
25%        60.269177
50%        66.583539
75%        83.298876
max       100.000000
Name: traffic_impact_index, dtype: float64


IMPACT CATEGORY

In [ ]:
def category(x):

    if x >= 80:
        return "Critical"

    elif x >= 60:
        return "High"

    elif x >= 40:
        return "Medium"

    return "Low"

df["impact_category"] = (

    df["traffic_impact_index"]

    .apply(category)

)

In [ ]:
print(

    df["impact_category"]

    .value_counts()

)

impact_category
High        3689
Critical    2815
Medium      1359
Low          310
Name: count, dtype: int64


FEATURE SELECTION

In [ ]:
features = [

    "event_type",
    "event_cause",
    "veh_type",
    "corridor",
    "zone",
    "junction",

    "latitude",
    "longitude",

    "duration_minutes",

    "hour",
    "weekday",
    "month",

    "weekend",
    "peak_hour",

    "event_frequency_score",

    "closure_risk",
    "duration_risk",
    "junction_count",
    "junction_duration",
    "corridor_count",
    "corridor_duration",
    "historical_impact"

]

TRAIN XGBOOST

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

X = df[features]

y = df["traffic_impact_index"]

cat_cols = [

    "event_type",
    "event_cause",
    "veh_type",
    "corridor",
    "zone",
    "junction"

]

num_cols = [

    c for c in features

    if c not in cat_cols

]

preprocessor = ColumnTransformer(

    [

        (

            "cat",

            OneHotEncoder(

                handle_unknown="ignore"

            ),

            cat_cols

        )

    ],

    remainder="passthrough"

)

X_train,X_test,y_train,y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42

)

model = XGBRegressor(

    n_estimators=500,

    max_depth=8,

    learning_rate=0.05,

    random_state=42

)

pipeline = Pipeline(

    [

        ("prep",preprocessor),

        ("model",model)

    ]

)

pipeline.fit(

    X_train,

    y_train

)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('prep',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['event_type', 'event_cause',
                                                   'veh_type', 'corridor',
                                                   'zone', 'junction'])])),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=None,
                              num_parallel_tree=None, ...))])

EVALUATION

In [ ]:
from sklearn.metrics import *

pred = pipeline.predict(X_test)

print(

    "R2:",

    r2_score(

        y_test,

        pred

    )

)

print(

    "MAE:",

    mean_absolute_error(

        y_test,

        pred

    )

)

R2: 0.9976933332369049
MAE: 0.0912896015217384


SAVE ARTIFACTS

In [ ]:
import joblib

joblib.dump(

    pipeline,

    "tii_model.pkl"

)

joblib.dump(

    features,

    "feature_columns.pkl"

)

print("Saved")

Saved


SAVE KNOWLEDGE BASE

In [ ]:
event_severity = {}

for e in df["event_type"].unique():

    subset = df[df["event_type"] == e]

    event_severity[e] = float(
        subset["traffic_impact_index"].mean()
    )

with open("event_severity.json","w") as f:
    import json
    json.dump(event_severity,f,indent=2)

In [ ]:
import json

with open(
    "junction_risk.json",
    "w"
) as f:
    json.dump(
        junction_risk,
        f,
        indent=2
    )

with open(
    "corridor_risk.json",
    "w"
) as f:
    json.dump(
        corridor_risk,
        f,
        indent=2
    )

with open(
    "event_severity.json",
    "w"
) as f:
    json.dump(
        event_severity,
        f,
        indent=2
    )

print("Risk Tables Saved")

Risk Tables Saved


In [ ]:
print(
    sorted(
        corridor_risk.items(),
        key=lambda x:x[1],
        reverse=True
    )[:10]
)

[('Non-corridor', 1576.346070195796), ('Mysore Road', 374.5101261664424), ('Bellary Road 1', 307.95016784153006), ('ORR East 2', 264.5848035650624), ('Tumkur Road', 232.66765822416303), ('Bellary Road 2', 197.16002702286718), ('ORR North 1', 150.448785), ('Hosur Road', 149.79246552013421), ('Old Madras Road', 141.51393374524716), ('ORR North 2', 131.23367113475177)]


In [ ]:
print(
    sorted(
        junction_risk.items(),
        key=lambda x:x[1],
        reverse=True
    )[:10]
)

[('Peenya2ndStageBusStand', 361.0), ('UpparpetJunction', 361.0), ('SadahalliGateJunc(AirportRd)', 249.09617023809525), ('Delmia-Jayanagar', 241.5), ('WebbsCircle', 241.5), ('ISRO Junction-NewBEL Rd', 236.65472499999998), ('Srinivagilu(Ejipura)Junc', 182.0), ('HennurRd-DavisRdJunction', 146.5), ('BEML GateJunc(SuranjandasRd)', 119.07736249999999), ('Geethanjali jn', 119.07736249999999)]


In [ ]:
import json
import joblib

# model
joblib.dump(pipeline, "tii_model.pkl")

# features
joblib.dump(features, "feature_columns.pkl")

# risk tables
with open("junction_risk.json","w") as f:
    json.dump(junction_risk,f,indent=2)

with open("corridor_risk.json","w") as f:
    json.dump(corridor_risk,f,indent=2)

with open("event_severity.json","w") as f:
    json.dump(event_severity,f,indent=2)

DOWNLOAD EVERYTHING

In [ ]:
from google.colab import files

files.download("tii_model.pkl")
files.download("feature_columns.pkl")
files.download("junction_risk.json")
files.download("corridor_risk.json")
files.download("event_severity.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>